# Warfarin preprocessing V2 — strict raw-feature candidate sets

This notebook builds the active V2 modeling table directly from `data/warfarin.csv` using only patient-side covariates. It keeps the two serious height/weight variants and adds raw clinical/genetic feature engineering: indication multi-hot features, comorbidity/medication text flags, genotype allele counts, target-INR numeric features, BMI/BSA, and clinically motivated interactions.

Formula-derived clinical/pharmacogenetic dose estimates are intentionally not used as model features.


## Height / weight imputation overview

The original preprocessing used KNN imputation on only two columns:

\[
X_{hw} = [\text{Height}, \text{Weight}]
\]

The values were standardized, imputed with `KNNImputer(n_neighbors=5)`, then inverse transformed. That is useful when one of height/weight is present, but when **both** are missing the imputer has almost no patient-specific signal and falls back toward global averages.

V2 keeps that old-style variant for comparison, but also creates two stronger alternatives:

1. **Group median imputation** using demographic groups such as gender, race, and age bucket.
2. **Regression-style multivariate imputation** using height, weight, age, gender, race, ethnicity, enzyme-inducer status, amiodarone status, and selected clinical/binary variables.

The regression imputer is the preferred V2 default because it uses patient context rather than only the two anthropometric columns.

In [1]:
from pathlib import Path
print('Notebook start cwd:', Path.cwd())

Notebook start cwd: /Users/dhillo/Garage/codeComplete/full measure/warfarin/src/preprocessing


In [2]:
# Path setup is handled robustly below.

In [3]:
# No manual os.chdir is needed here.

In [4]:
from pathlib import Path
import os
import sys
import json
import re

import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

In [5]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "data" / "warfarin.csv").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing data/warfarin.csv and src/.")


REPO_ROOT = find_repo_root()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [6]:
RAW_PATH = REPO_ROOT / "data" / "warfarin.csv"
OUTPUT_DIR = REPO_ROOT / "output" / "preprocess_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "Therapeutic Dose of Warfarin"
TRUE_DOSE_COL = "Therapeutic Dose of Warfarin__mg_week"
DOSE_BINS = [0, 20.9999, 49, np.inf]
DOSE_LABELS = [0, 1, 2]

print("Repo root:", REPO_ROOT)
print("Raw data path:", RAW_PATH)
print("V2 output directory:", OUTPUT_DIR)

Repo root: /Users/dhillo/Garage/codeComplete/full measure/warfarin
Raw data path: /Users/dhillo/Garage/codeComplete/full measure/warfarin/data/warfarin.csv
V2 output directory: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2


## Configuration

The output file names are unchanged so downstream notebooks can keep using the same V2 paths. The feature manifest now contains only the serious candidates.


In [7]:
ID_COLUMNS = ["PharmGKB Subject ID"]
EMPTY_COLUMNS = ["Unnamed: 63", "Unnamed: 64", "Unnamed: 65"]

POST_TREATMENT_OR_OUTCOME_COLUMNS = [
    "Subject Reached Stable Dose of Warfarin",
    "INR on Reported Therapeutic Dose of Warfarin",
]

FREE_TEXT_COLUMNS = [
    "Comorbidities",
    "Medications",
]

# For these medication statuses, the appendix-style preprocessing treats unknown as not used.
ZERO_IF_UNKNOWN_COLUMNS = [
    "Amiodarone (Cordarone)",
    "Carbamazepine (Tegretol)",
    "Phenytoin (Dilantin)",
    "Rifampin or Rifampicin",
]

# Other 0/1 clinical/medication columns where missingness is better treated as an explicit category.
BINARY_UNKNOWN_COLUMNS = [
    "Diabetes",
    "Congestive Heart Failure and/or Cardiomyopathy",
    "Valve Replacement",
    "Aspirin",
    "Acetaminophen or Paracetamol (Tylenol)",
    "Was Dose of Acetaminophen or Paracetamol (Tylenol) >1300mg/day",
    "Simvastatin (Zocor)",
    "Atorvastatin (Lipitor)",
    "Fluvastatin (Lescol)",
    "Lovastatin (Mevacor)",
    "Pravastatin (Pravachol)",
    "Rosuvastatin (Crestor)",
    "Cerivastatin (Baycol)",
    "Sulfonamide Antibiotics",
    "Macrolide Antibiotics",
    "Anti-fungal Azoles",
    "Herbal Medications, Vitamins, Supplements",
    "Current Smoker",
]

GENOTYPE_COLUMNS = [
    "Cyp2C9 genotypes",
    "Genotyped QC Cyp2C9*2",
    "Genotyped QC Cyp2C9*3",
    "Combined QC CYP2C9",
    "VKORC1 genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T",
    "VKORC1 QC genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T",
    "VKORC1 genotype: 497T>G (5808); chr16:31013055; rs2884737; A/C",
    "VKORC1 QC genotype: 497T>G (5808); chr16:31013055; rs2884737; A/C",
    "VKORC1 genotype: 1173 C>T(6484); chr16:31012379; rs9934438; A/G",
    "VKORC1 QC genotype: 1173 C>T(6484); chr16:31012379; rs9934438; A/G",
    "VKORC1 genotype: 1542G>C (6853); chr16:31012010; rs8050894; C/G",
    "VKORC1 QC genotype: 1542G>C (6853); chr16:31012010; rs8050894; C/G",
    "VKORC1 genotype: 3730 G>A (9041); chr16:31009822; rs7294;  A/G",
    "VKORC1 QC genotype: 3730 G>A (9041); chr16:31009822; rs7294;  A/G",
    "VKORC1 genotype: 2255C>T (7566); chr16:31011297; rs2359612; A/G",
    "VKORC1 QC genotype: 2255C>T (7566); chr16:31011297; rs2359612; A/G",
    "VKORC1 genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C",
    "VKORC1 QC genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C",
    "CYP2C9 consensus",
    "VKORC1 -1639 consensus",
    "VKORC1 497 consensus",
    "VKORC1 1173 consensus",
    "VKORC1 1542 consensus",
    "VKORC1 3730 consensus",
    "VKORC1 2255 consensus",
    "VKORC1 -4451 consensus",
]

AGE_TO_DECADE = {
    "0 - 9": 0,
    "10 - 19": 1,
    "20 - 29": 2,
    "30 - 39": 3,
    "40 - 49": 4,
    "50 - 59": 5,
    "60 - 69": 6,
    "70 - 79": 7,
    "80 - 89": 8,
    "90+": 9,
}

INDICATION_MAP = {
    "1": "DVT",
    "2": "PE",
    "3": "Afib_or_flutter",
    "4": "Heart_Valve",
    "5": "Cardiomyopathy_or_LV_Dilation",
    "6": "Stroke",
    "7": "Post_Orthopedic",
    "8": "Other",
}

COMORBIDITY_PATTERNS = {
    "Comorbidity__atrial_fibrillation_or_flutter": [r"\batrial\s+fibril+ation\b", r"\batrial\s+fibrillation\b", r"\bafib\b", r"\bflutter\b"],
    "Comorbidity__heart_valve": [r"\bvalve\b", r"\bprosthe(?:sis|tic)\b", r"\bmitral\b", r"\baortic\b"],
    "Comorbidity__thrombosis_or_embolism": [r"\bthrombo", r"\bembol", r"\bdvt\b", r"\bpulmonary\s+embol"],
    "Comorbidity__cardiomyopathy_or_heart_failure": [r"\bcardiomyopathy\b", r"\bheart\s+failure\b", r"\bchf\b", r"\blv\s+dilation\b"],
    "Comorbidity__diabetes": [r"\bdiabetes\b", r"\bdiabetic\b"],
    "Comorbidity__hypertension": [r"\bhypertension\b", r"\bhigh\s+blood\s+pressure\b"],
    "Comorbidity__hyperlipidemia": [r"\bhyperlipid", r"\bdyslipid"],
    "Comorbidity__cancer": [r"\bcancer\b", r"\bmalignan", r"\bcarcinoma\b", r"\blymphoma\b", r"\bleukemia\b"],
    "Comorbidity__renal_or_kidney": [r"\brenal\b", r"\bkidney\b", r"\bckd\b", r"\bdialysis\b"],
    "Comorbidity__liver_or_hepatic": [r"\bliver\b", r"\bhepatic\b", r"\bcirrhos"],
    "Comorbidity__stroke_or_cva": [r"\bstroke\b", r"\bcva\b", r"\bcerebrovascular\b"],
    "Comorbidity__obesity": [r"\bobes"],
    "Comorbidity__respiratory": [r"\brespiratory\b", r"\bcopd\b", r"\basthma\b"],
    "Comorbidity__cardiovascular": [r"\bcardiovascular\b", r"\bcoronary\b", r"\bcad\b", r"\bmyocardial\b", r"\barr?ythmia\b"],
}

MEDICATION_PATTERNS = {
    "MedicationText__aspirin": [r"\baspirin\b"],
    "MedicationText__clopidogrel_or_antiplatelet": [r"\bclopidogrel\b", r"\bplavix\b", r"\bdipyridamole\b", r"\bantiplatelet\b"],
    "MedicationText__statin": [r"\bstatin\b", r"\bsimvastatin\b", r"\batorvastatin\b", r"\bpravastatin\b", r"\brosuvastatin\b", r"\blovastatin\b", r"\bfluvastatin\b"],
    "MedicationText__amiodarone": [r"\bamiodarone\b", r"\bcordarone\b", r"\bpacerone\b"],
    "MedicationText__acetaminophen": [r"\bacetaminophen\b", r"\bparacetamol\b", r"\btylenol\b"],
    "MedicationText__antibiotic": [r"\bantibiotic\b", r"\bpenicillin\b", r"\bsulfa", r"\bmacrolide\b", r"\bazithro", r"\bclarithro", r"\berythro"],
    "MedicationText__antifungal_azole": [r"\bantifungal\b", r"\bazole\b", r"\bfluconazole\b", r"\bketoconazole\b", r"\bvoriconazole\b", r"\bitraconazole\b"],
    "MedicationText__nsaid_or_cox2": [r"\bnsaid\b", r"\bibuprofen\b", r"\bnaproxen\b", r"\bcelebrex\b", r"\bcelecoxib\b", r"\bvioxx\b", r"\bbextra\b"],
    "MedicationText__herbal_vitamin_supplement": [r"\bherbal\b", r"\bvitamin\b", r"\bsupplement\b", r"\bst\.?\s*john"],
    "MedicationText__enzyme_inducer": [r"\bcarbamazepine\b", r"\btegretol\b", r"\bphenytoin\b", r"\bdilantin\b", r"\brifampin\b", r"\brifampicin\b"],
    "MedicationText__cyp2c9_substrate_or_flag": [r"\bsubstrate2c9\b", r"\bcyp2c9\b"],
}


## Load raw data and inspect missingness

In [8]:
df_raw = pd.read_csv(RAW_PATH)
print("Raw shape:", df_raw.shape)

missing_report_raw = (
    df_raw.isna().sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_report_raw["missing_pct"] = missing_report_raw["missing_count"] / len(df_raw)
missing_report_raw = missing_report_raw.sort_values("missing_count", ascending=False)

missing_report_path = OUTPUT_DIR / "warfarin_preprocess_v2_missingness_report_raw.csv"
missing_report_raw.to_csv(missing_report_path, index=False)

print("Saved raw missingness report:", missing_report_path)
missing_report_raw.head(30)

Raw shape: (5701, 66)
Saved raw missingness report: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_missingness_report_raw.csv


,column,missing_count,missing_pct
65,Unnamed: 65,5701,1.000000
64,Unnamed: 64,5701,1.000000
63,Unnamed: 63,5701,1.000000
54,VKORC1 QC genotype: -4451 C>A (861); Chr16:310...,5250,0.920891
48,VKORC1 QC genotype: 1542G>C (6853); chr16:3101...,5223,0.916155
39,Genotyped QC Cyp2C9*3,5223,0.916155
40,Combined QC CYP2C9,5223,0.916155
42,VKORC1 QC genotype: -1639 G>A (3673); chr16:31...,5223,0.916155
44,VKORC1 QC genotype: 497T>G (5808); chr16:31013...,5223,0.916155
46,VKORC1 QC genotype: 1173 C>T(6484); chr16:3101...,5223,0.916155


In [9]:
# Keep only patients with known therapeutic dose, as specified by the project.
df = df_raw[df_raw[TARGET_COL].notna()].copy()
df[TRUE_DOSE_COL] = df[TARGET_COL].astype(float)
df[TARGET_COL] = pd.cut(
    df[TRUE_DOSE_COL],
    bins=DOSE_BINS,
    labels=DOSE_LABELS,
    include_lowest=True,
).astype(int)

print("Shape after dropping unknown therapeutic dose:", df.shape)
print("Dose bucket distribution:")
print(df[TARGET_COL].value_counts().sort_index())

Shape after dropping unknown therapeutic dose: (5528, 67)
Dose bucket distribution:
Therapeutic Dose of Warfarin
0    1495
1    3382
2     651
Name: count, dtype: int64


## Height / weight missingness statistics

These counts help us understand why the original two-column KNN imputation is limited. When both height and weight are missing, a two-column KNN imputer cannot use patient-level context.

In [10]:
height_missing = df["Height (cm)"].isna()
weight_missing = df["Weight (kg)"].isna()

height_weight_missing_stats = pd.DataFrame(
    [
        {"case": "height_present_weight_present", "count": int((~height_missing & ~weight_missing).sum())},
        {"case": "height_missing_weight_present", "count": int((height_missing & ~weight_missing).sum())},
        {"case": "height_present_weight_missing", "count": int((~height_missing & weight_missing).sum())},
        {"case": "height_missing_weight_missing", "count": int((height_missing & weight_missing).sum())},
    ]
)
height_weight_missing_stats["pct"] = height_weight_missing_stats["count"] / len(df)
height_weight_missing_stats_path = OUTPUT_DIR / "warfarin_preprocess_v2_height_weight_missingness.csv"
height_weight_missing_stats.to_csv(height_weight_missing_stats_path, index=False)

print("Saved height/weight missingness stats:", height_weight_missing_stats_path)
height_weight_missing_stats

Saved height/weight missingness stats: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_height_weight_missingness.csv


,case,count,pct
0,height_present_weight_present,4417,0.799023
1,height_missing_weight_present,839,0.151773
2,height_present_weight_missing,30,0.005427
3,height_missing_weight_missing,242,0.043777


## Core cleaning helpers

In [11]:
def yes_no_unknown(series):
    return series.map({1.0: "Yes", 0.0: "No", 1: "Yes", 0: "No"}).fillna("Unknown")


def yes_indicator(series):
    return series.astype(str).str.strip().str.lower().isin(["1", "1.0", "yes", "true"]).astype(float)


def parse_indication_codes(value):
    if pd.isna(value):
        return []
    text = str(value).lower().replace("or", ";").replace(",", ";")
    parts = [part.strip() for part in text.split(";")]
    return [part for part in parts if part in INDICATION_MAP]


def add_indication_features(frame):
    parsed = frame["Indication for Warfarin Treatment"].apply(parse_indication_codes)
    for code, name in INDICATION_MAP.items():
        frame[f"Indication__{name}"] = parsed.apply(lambda codes, c=code: int(c in codes))
    frame["Indication__Unknown"] = parsed.apply(lambda codes: int(len(codes) == 0))
    frame["Indication__Multiple"] = parsed.apply(lambda codes: int(len(codes) > 1))
    frame["Indication__Count"] = parsed.apply(len).astype(float)
    return frame


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).lower()
    text = re.sub(r"[^a-z0-9*+./;,_ -]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_text_chunks(value):
    text = normalize_text(value)
    if not text:
        return []
    return [chunk.strip() for chunk in re.split(r"[;,]|\band/or\b|\band\b|\bor\b", text) if chunk.strip()]


def chunk_is_negated(chunk):
    return bool(re.search(r"\b(no|not|none|without|denies|negative for)\b", chunk))


def has_positive_pattern(value, patterns):
    for chunk in split_text_chunks(value):
        if chunk_is_negated(chunk):
            continue
        if any(re.search(pattern, chunk) for pattern in patterns):
            return 1
    return 0


def add_text_pattern_features(frame, source_col, pattern_map, missing_col, none_col):
    text = frame[source_col]
    normalized = text.apply(normalize_text)
    frame[missing_col] = text.isna().astype(int)
    frame[none_col] = normalized.str.fullmatch(r"(|none|no|nil|na|n/a|no comorbidities|no interacting medications)").fillna(False).astype(int)
    for feature_name, patterns in pattern_map.items():
        frame[feature_name] = text.apply(lambda value, p=patterns: has_positive_pattern(value, p)).astype(int)
    return frame


def parse_target_inr_value(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"\d+(?:\.\d+)?", str(value))
    return float(match.group(0)) if match else np.nan


def parse_target_inr_range(value):
    if pd.isna(value):
        return (np.nan, np.nan)
    nums = re.findall(r"\d+(?:\.\d+)?", str(value))
    if len(nums) >= 2:
        low, high = float(nums[0]), float(nums[1])
        return (min(low, high), max(low, high))
    if len(nums) == 1:
        val = float(nums[0])
        return (val, val)
    return (np.nan, np.nan)


def add_target_inr_features(frame):
    """Parse target INR information without using reported therapeutic INR leakage."""
    exact = frame["Target INR"].apply(parse_target_inr_value)
    ranges = frame["Estimated Target INR Range Based on Indication"].apply(parse_target_inr_range)
    lower = ranges.apply(lambda x: x[0])
    upper = ranges.apply(lambda x: x[1])
    midpoint = (lower + upper) / 2.0
    width = upper - lower

    target_from_exact_or_range = exact.combine_first(midpoint)
    default_target = float(target_from_exact_or_range.median())
    if pd.isna(default_target):
        default_target = 2.5

    frame["TargetINR__exact_missing"] = exact.isna().astype(int)
    frame["TargetINR__range_missing"] = midpoint.isna().astype(int)
    frame["TargetINR__any_available"] = (~target_from_exact_or_range.isna()).astype(int)
    frame["TargetINR__value"] = target_from_exact_or_range.fillna(default_target).astype(float)
    frame["TargetINR__range_lower"] = lower.combine_first(exact).fillna(default_target).astype(float)
    frame["TargetINR__range_upper"] = upper.combine_first(exact).fillna(default_target).astype(float)
    frame["TargetINR__range_midpoint"] = midpoint.combine_first(exact).fillna(default_target).astype(float)
    frame["TargetINR__range_width"] = width.fillna(0.0).clip(lower=0).astype(float)
    frame["TargetINR__standard_2_to_3"] = ((lower == 2.0) & (upper == 3.0)).astype(int)
    frame["TargetINR__high_intensity"] = (frame["TargetINR__range_upper"] >= 3.5).astype(int)
    frame["TargetINR__low_intensity"] = (frame["TargetINR__range_lower"] < 2.0).astype(int)
    return frame


def count_star_allele(genotype, allele):
    if pd.isna(genotype) or str(genotype).strip().lower() == "unknown":
        return 0
    alleles = re.findall(r"\*\d+", str(genotype))
    return int(sum(a == allele for a in alleles))


def count_reduced_cyp2c9_alleles(genotype):
    if pd.isna(genotype) or str(genotype).strip().lower() == "unknown":
        return 0
    alleles = re.findall(r"\*\d+", str(genotype))
    return int(sum(a != "*1" for a in alleles))


def count_vkorc1_allele(genotype, allele):
    if pd.isna(genotype) or str(genotype).strip().lower() == "unknown":
        return 0
    alleles = re.split(r"[/|]", str(genotype).strip())
    return int(sum(a.strip().upper() == allele for a in alleles))


def add_genotype_dosage_features(frame):
    cyp = frame["Cyp2C9 genotypes"]
    frame["CYP2C9__star2_count"] = cyp.apply(lambda value: count_star_allele(value, "*2")).astype(float)
    frame["CYP2C9__star3_count"] = cyp.apply(lambda value: count_star_allele(value, "*3")).astype(float)
    frame["CYP2C9__reduced_allele_count"] = cyp.apply(count_reduced_cyp2c9_alleles).astype(float)
    frame["CYP2C9__rare_allele_count"] = (
        frame["CYP2C9__reduced_allele_count"]
        - frame["CYP2C9__star2_count"]
        - frame["CYP2C9__star3_count"]
    ).clip(lower=0).astype(float)
    frame["CYP2C9__unknown"] = cyp.astype(str).str.lower().eq("unknown").astype(int)

    vkorc1_col = "VKORC1 genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T"
    vkorc1 = frame[vkorc1_col]
    frame["VKORC1_rs9923231__A_count"] = vkorc1.apply(lambda value: count_vkorc1_allele(value, "A")).astype(float)
    frame["VKORC1_rs9923231__G_count"] = vkorc1.apply(lambda value: count_vkorc1_allele(value, "G")).astype(float)
    frame["VKORC1_rs9923231__unknown"] = vkorc1.astype(str).str.lower().eq("unknown").astype(int)
    return frame


def race_flag(series, keyword):
    return series.astype(str).str.lower().str.contains(keyword, regex=False).astype(float)


def add_body_size_features(frame, variant_name, height_col, weight_col):
    height_m = frame[height_col].astype(float) / 100.0
    weight = frame[weight_col].astype(float)
    frame[f"BMI__{variant_name}"] = (weight / np.square(height_m)).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    frame[f"BSA_Mosteller__{variant_name}"] = np.sqrt((frame[height_col].astype(float) * weight) / 3600.0)
    return frame


def add_interaction_features(frame, variant_name, height_col, weight_col):
    """Small set of clinically plausible raw-feature interactions for linear bandits."""
    age = frame["Age__mode_decade"].astype(float)
    weight = frame[weight_col].astype(float)
    height = frame[height_col].astype(float)
    bmi = frame[f"BMI__{variant_name}"].astype(float)
    bsa = frame[f"BSA_Mosteller__{variant_name}"].astype(float)
    cyp_reduced = frame["CYP2C9__reduced_allele_count"].astype(float)
    cyp_star3 = frame["CYP2C9__star3_count"].astype(float)
    vkorc1_a = frame["VKORC1_rs9923231__A_count"].astype(float)
    amiodarone = frame["Amiodarone (Cordarone)"].astype(float)
    enzyme = frame["Enzyme inducer status"].astype(float)
    smoker = yes_indicator(frame.get("Current Smoker", pd.Series("Unknown", index=frame.index)))
    valve_replacement = yes_indicator(frame.get("Valve Replacement", pd.Series("Unknown", index=frame.index)))
    race_asian = race_flag(frame["Race"], "asian")
    race_black = race_flag(frame["Race"], "black")
    target_mid = frame["TargetINR__range_midpoint"].astype(float)

    interactions = {
        f"Interaction__AgeDecade_x_Weight__{variant_name}": age * weight,
        f"Interaction__AgeDecade_x_BMI__{variant_name}": age * bmi,
        f"Interaction__BSA_x_VKORC1_Acount__{variant_name}": bsa * vkorc1_a,
        f"Interaction__Weight_x_EnzymeInducer__{variant_name}": weight * enzyme,
        f"Interaction__Weight_x_CYP2C9Reduced__{variant_name}": weight * cyp_reduced,
        f"Interaction__Height_x_CYP2C9Reduced__{variant_name}": height * cyp_reduced,
        "Interaction__AgeDecade_x_CYP2C9Reduced": age * cyp_reduced,
        "Interaction__AgeDecade_x_CYP2C9Star3": age * cyp_star3,
        "Interaction__AgeDecade_x_VKORC1_Acount": age * vkorc1_a,
        "Interaction__Amiodarone_x_CYP2C9Reduced": amiodarone * cyp_reduced,
        "Interaction__Amiodarone_x_VKORC1_Acount": amiodarone * vkorc1_a,
        "Interaction__EnzymeInducer_x_CYP2C9Reduced": enzyme * cyp_reduced,
        "Interaction__EnzymeInducer_x_VKORC1_Acount": enzyme * vkorc1_a,
        "Interaction__CurrentSmoker_x_AgeDecade": smoker * age,
        "Interaction__RaceAsian_x_VKORC1_Acount": race_asian * vkorc1_a,
        "Interaction__RaceAsian_x_CYP2C9Reduced": race_asian * cyp_reduced,
        "Interaction__RaceBlack_x_VKORC1_Acount": race_black * vkorc1_a,
        "Interaction__RaceBlack_x_CYP2C9Reduced": race_black * cyp_reduced,
        "Interaction__HeartValveIndication_x_TargetINR": frame["Indication__Heart_Valve"].astype(float) * target_mid,
        "Interaction__ValveReplacement_x_TargetINR": valve_replacement * target_mid,
        "Interaction__HighIntensityINR_x_ValveIndication": frame["TargetINR__high_intensity"].astype(float) * frame["Indication__Heart_Valve"].astype(float),
    }
    for name, values in interactions.items():
        if name not in frame.columns:
            frame[name] = pd.Series(values, index=frame.index).astype(float)
    return frame


def save_json(data, path):
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)


## Apply V2 cleaning rules

In [12]:
# Drop empty columns but keep the subject ID in the raw-clean table for traceability.
df = df.drop(columns=[c for c in EMPTY_COLUMNS if c in df.columns], errors="ignore")

# Demographics
for col in ["Gender", "Race", "Ethnicity"]:
    df[col] = df[col].fillna("Unknown")

mode_age_bucket = df["Age"].mode(dropna=True).iloc[0]
df["Age__mode_bucket"] = df["Age"].fillna(mode_age_bucket)
df["Age__bucket_unknown"] = df["Age"].fillna("Unknown")
df["Age__missing"] = df["Age"].isna().astype(int)
df["Age__mode_decade"] = df["Age__mode_bucket"].map(AGE_TO_DECADE).astype(float)

print("Age mode bucket used for numeric age:", mode_age_bucket)
print(df[["Age", "Age__mode_bucket", "Age__bucket_unknown", "Age__mode_decade"]].head())

Age mode bucket used for numeric age: 70 - 79
       Age Age__mode_bucket Age__bucket_unknown  Age__mode_decade
0  60 - 69          60 - 69             60 - 69               6.0
1  50 - 59          50 - 59             50 - 59               5.0
2  40 - 49          40 - 49             40 - 49               4.0
3  60 - 69          60 - 69             60 - 69               6.0
4  50 - 59          50 - 59             50 - 59               5.0


In [13]:
# Appendix-guided medication statuses: unknown is treated as not used for these specific columns.
for col in ZERO_IF_UNKNOWN_COLUMNS:
    if col in df.columns:
        df[f"{col}__missing"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(0).astype(float)

# Enzyme inducer status is defined by carbamazepine, phenytoin, or rifampin/rifampicin use.
df["Enzyme inducer status"] = (
    (df["Carbamazepine (Tegretol)"] == 1)
    | (df["Phenytoin (Dilantin)"] == 1)
    | (df["Rifampin or Rifampicin"] == 1)
).astype(float)

# Other binary variables keep Unknown as its own category.
for col in BINARY_UNKNOWN_COLUMNS:
    if col in df.columns:
        df[col] = yes_no_unknown(df[col])

print("Enzyme inducer status value counts:")
print(df["Enzyme inducer status"].value_counts(dropna=False))

Enzyme inducer status value counts:
Enzyme inducer status
0.0    5474
1.0      54
Name: count, dtype: int64


## VKORC1 rs9923231 imputation from appendix S4

The appendix gives a decision list for imputing VKORC1 rs9923231 from nearby VKORC1 SNPs and race. V2 keeps that logic, but treats the dataset value `Unknown` as the missing/mixed-race category for the race-gated conditions.

In [14]:
def impute_vkorc1_rs9923231(frame):
    frame = frame.copy()
    target = "VKORC1 genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T"
    rs2359612 = "VKORC1 genotype: 2255C>T (7566); chr16:31011297; rs2359612; A/G"
    rs9934438 = "VKORC1 genotype: 1173 C>T(6484); chr16:31012379; rs9934438; A/G"
    rs8050894 = "VKORC1 genotype: 1542G>C (6853); chr16:31012010; rs8050894; C/G"

    before_missing = int(frame[target].isna().sum())
    race_blocks_race_gated_imputation = frame["Race"].isin([
        "Black or African American",
        "Missing or Mixed Race",
        "Unknown",
    ])
    race_gated = ~race_blocks_race_gated_imputation

    rules = [
        (race_gated & (frame[rs2359612] == "C/C"), "G/G"),
        (race_gated & (frame[rs2359612] == "T/T"), "A/A"),
        (race_gated & (frame[rs2359612] == "C/T"), "A/G"),
        ((frame[rs9934438] == "C/C"), "G/G"),
        ((frame[rs9934438] == "T/T"), "A/A"),
        ((frame[rs9934438] == "C/T"), "A/G"),
        (race_gated & (frame[rs8050894] == "G/G"), "G/G"),
        (race_gated & (frame[rs8050894] == "C/C"), "A/A"),
        (race_gated & (frame[rs8050894] == "C/G"), "A/G"),
    ]

    for condition, value in rules:
        frame.loc[condition & frame[target].isna(), target] = value

    after_rules_missing = int(frame[target].isna().sum())
    frame[target] = frame[target].fillna("Unknown")
    after_final_missing = int(frame[target].isna().sum())

    report = {
        "target_column": target,
        "missing_before": before_missing,
        "missing_after_decision_rules": after_rules_missing,
        "missing_after_unknown_fill": after_final_missing,
        "imputed_by_decision_rules": before_missing - after_rules_missing,
    }
    return frame, report


df, vkorc1_report = impute_vkorc1_rs9923231(df)

# Fill remaining genotype and consensus missingness as Unknown.
for col in GENOTYPE_COLUMNS:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

vkorc1_report_path = OUTPUT_DIR / "warfarin_preprocess_v2_vkorc1_imputation_report.json"
save_json(vkorc1_report, vkorc1_report_path)

print("Saved VKORC1 imputation report:", vkorc1_report_path)
print(vkorc1_report)

Saved VKORC1 imputation report: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_vkorc1_imputation_report.json
{'target_column': 'VKORC1 genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T', 'missing_before': 1633, 'missing_after_decision_rules': 124, 'missing_after_unknown_fill': 0, 'imputed_by_decision_rules': 1509}


## Raw clinical text, target-INR, indication, genotype-dosage, and interaction features

In [15]:
# Indication is a legitimate pre-treatment clinical context column.
df = add_indication_features(df)

# Target INR/range can be known before initial dose choice and is not the same as reported therapeutic INR.
df = add_target_inr_features(df)

# Free-text columns contain useful clinical signal that was previously dropped.
df = add_text_pattern_features(
    df,
    source_col="Comorbidities",
    pattern_map=COMORBIDITY_PATTERNS,
    missing_col="ComorbidityText__missing",
    none_col="ComorbidityText__none_reported",
)
df = add_text_pattern_features(
    df,
    source_col="Medications",
    pattern_map=MEDICATION_PATTERNS,
    missing_col="MedicationText__missing",
    none_col="MedicationText__none_reported",
)

# Compact numeric genotype dosage features help the linear bandits use clinically meaningful allele counts.
df = add_genotype_dosage_features(df)

engineered_prefixes = [
    "Indication__",
    "TargetINR__",
    "Comorbidity__",
    "ComorbidityText__",
    "MedicationText__",
    "CYP2C9__",
    "VKORC1_rs9923231__",
]
engineered_feature_report = []
for prefix in engineered_prefixes:
    cols = [col for col in df.columns if col.startswith(prefix)]
    for col in cols:
        values = pd.to_numeric(df[col], errors="coerce")
        engineered_feature_report.append({
            "feature": col,
            "mean_or_rate": float(values.mean()),
            "nonzero_count": int((values.fillna(0) != 0).sum()),
        })

engineered_feature_report = pd.DataFrame(engineered_feature_report).sort_values("feature")
engineered_feature_report_path = OUTPUT_DIR / "warfarin_preprocess_v2_engineered_raw_feature_report.csv"
engineered_feature_report.to_csv(engineered_feature_report_path, index=False)

print("Saved engineered raw feature report:", engineered_feature_report_path)
print("New engineered raw features before body-size/interactions:", len(engineered_feature_report))
engineered_feature_report.head(40)


Saved engineered raw feature report: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_engineered_raw_feature_report.csv
New engineered raw features before body-size/interactions: 59


,feature,mean_or_rate,nonzero_count
54,CYP2C9__rare_allele_count,0.003075,17
53,CYP2C9__reduced_allele_count,0.270803,1353
51,CYP2C9__star2_count,0.161360,836
52,CYP2C9__star3_count,0.106368,568
55,CYP2C9__unknown,0.019718,109
36,ComorbidityText__missing,0.350941,1940
37,ComorbidityText__none_reported,0.417149,2306
22,Comorbidity__atrial_fibrillation_or_flutter,0.208936,1155
29,Comorbidity__cancer,0.045767,253
25,Comorbidity__cardiomyopathy_or_heart_failure,0.090268,499


## Height / weight imputation variants, BMI/BSA, and clinical interactions

In [16]:
# Missingness flags are useful even after imputation.
df["Height (cm)__missing"] = df["Height (cm)"].isna().astype(int)
df["Weight (kg)__missing"] = df["Weight (kg)"].isna().astype(int)
df["HeightWeight__both_missing"] = (
    df["Height (cm)"].isna() & df["Weight (kg)"].isna()
).astype(int)

# Variant 1: old-style KNN on only height and weight.
# Kept only as a reference feature set, because regression imputation performed better overall.
hw_scaler = StandardScaler()
hw_scaled = hw_scaler.fit_transform(df[["Height (cm)", "Weight (kg)"]])
hw_knn = KNNImputer(n_neighbors=5).fit_transform(hw_scaled)
hw_knn = hw_scaler.inverse_transform(hw_knn)
df["Height (cm)__knn_hw"] = hw_knn[:, 0]
df["Weight (kg)__knn_hw"] = hw_knn[:, 1]

# Variant 2: multivariate regression-style imputation with demographics and key binary covariates.
hw_regression_source = pd.DataFrame(index=df.index)
hw_regression_source["Height (cm)"] = df["Height (cm)"]
hw_regression_source["Weight (kg)"] = df["Weight (kg)"]
hw_regression_source["Age__mode_decade"] = df["Age__mode_decade"]
hw_regression_source["Gender__male"] = df["Gender"].astype(str).str.lower().eq("male").astype(float)
hw_regression_source["Gender__female"] = df["Gender"].astype(str).str.lower().eq("female").astype(float)
hw_regression_source["Race__Asian"] = df["Race"].astype(str).str.lower().str.contains("asian", regex=False).astype(float)
hw_regression_source["Race__Black"] = df["Race"].astype(str).str.lower().str.contains("black", regex=False).astype(float)
hw_regression_source["Race__White"] = df["Race"].astype(str).str.lower().str.contains("white", regex=False).astype(float)
hw_regression_source["Diabetes__yes"] = yes_indicator(df.get("Diabetes", pd.Series("Unknown", index=df.index)))
hw_regression_source["CHF__yes"] = yes_indicator(df.get("Congestive Heart Failure and/or Cardiomyopathy", pd.Series("Unknown", index=df.index)))
hw_regression_source["CurrentSmoker__yes"] = yes_indicator(df.get("Current Smoker", pd.Series("Unknown", index=df.index)))

iterative_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42,
    initial_strategy="median",
    sample_posterior=False,
)
hw_regression_imputed = iterative_imputer.fit_transform(hw_regression_source)
hw_regression_imputed = pd.DataFrame(
    hw_regression_imputed,
    columns=hw_regression_source.columns,
    index=df.index,
)
df["Height (cm)__regression"] = hw_regression_imputed["Height (cm)"]
df["Weight (kg)__regression"] = hw_regression_imputed["Weight (kg)"]

# Derived body-size features and interactions are created separately per height/weight variant.
for variant_name, height_col, weight_col in [
    ("knn_hw", "Height (cm)__knn_hw", "Weight (kg)__knn_hw"),
    ("regression_hw", "Height (cm)__regression", "Weight (kg)__regression"),
]:
    df = add_body_size_features(df, variant_name, height_col, weight_col)
    df = add_interaction_features(df, variant_name, height_col, weight_col)

height_weight_variant_summary = pd.DataFrame([
    {
        "variant": "knn_hw",
        "height_mean": df["Height (cm)__knn_hw"].mean(),
        "height_std": df["Height (cm)__knn_hw"].std(),
        "weight_mean": df["Weight (kg)__knn_hw"].mean(),
        "weight_std": df["Weight (kg)__knn_hw"].std(),
    },
    {
        "variant": "regression_hw",
        "height_mean": df["Height (cm)__regression"].mean(),
        "height_std": df["Height (cm)__regression"].std(),
        "weight_mean": df["Weight (kg)__regression"].mean(),
        "weight_std": df["Weight (kg)__regression"].std(),
    },
])

height_weight_variant_path = OUTPUT_DIR / "warfarin_preprocess_v2_height_weight_variant_summary.csv"
height_weight_variant_summary.to_csv(height_weight_variant_path, index=False)
print("Saved height/weight variant summary:", height_weight_variant_path)

interaction_cols = [col for col in df.columns if col.startswith("Interaction__")]
body_cols = [col for col in df.columns if col.startswith("BMI__") or col.startswith("BSA_Mosteller__")]
for col in body_cols + interaction_cols:
    values = pd.to_numeric(df[col], errors="coerce")
    engineered_feature_report.loc[len(engineered_feature_report)] = {
        "feature": col,
        "mean_or_rate": float(values.mean()),
        "nonzero_count": int((values.fillna(0) != 0).sum()),
    }
engineered_feature_report = engineered_feature_report.sort_values("feature")
engineered_feature_report.to_csv(engineered_feature_report_path, index=False)
print("Body-size features:", len(body_cols))
print("Interaction features:", len(interaction_cols))
height_weight_variant_summary


Saved height/weight variant summary: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_height_weight_variant_summary.csv
Body-size features: 4
Interaction features: 27


/var/folders/zd/20hx99216gn2st0b9xbkh3n00000gn/T/ipykernel_70184/2274794242.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[name] = pd.Series(values, index=frame.index).astype(float)
/var/folders/zd/20hx99216gn2st0b9xbkh3n00000gn/T/ipykernel_70184/2274794242.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[name] = pd.Series(values, index=frame.index).astype(float)
/var/folders/zd/20hx99216gn2st0b9xbkh3n00000gn/T/ipykernel_70184/2274794242.py:210: PerformanceWarning: DataFrame is highly fragmented.  This is

,variant,height_mean,height_std,weight_mean,weight_std
0,knn_hw,167.856781,10.251407,77.796891,21.437094
1,regression_hw,167.632985,10.458577,77.699152,21.602018


## Clinical and pharmacogenetic algorithm accuracy by feature set

The fixed IWPC clinical and pharmacogenetic formulas are evaluated with the same cleaned patient attributes used by each V2 feature set. The only input that changes between the two evaluations is the height/weight imputation variant associated with that feature set.

These formula predictions are reporting-only baselines. They are not added to `df`, the modeling table, or either feature manifest.

In [17]:
def race_coefficient_clinical(race):
    if race == "Asian":
        return -0.6752
    if race == "Black or African American":
        return 0.4060
    if race in ["Unknown", "Missing or Mixed Race"]:
        return 0.0443
    return 0.0


def race_coefficient_pharmacogenetic(race):
    if race == "Asian":
        return -0.1092
    if race == "Black or African American":
        return -0.2760
    if race in ["Unknown", "Missing or Mixed Race"]:
        return -0.1032
    return 0.0


def cyp2c9_coefficient(genotype):
    return {
        "*1/*2": -0.5211,
        "*1/*3": -0.9357,
        "*2/*2": -1.0616,
        "*2/*3": -1.9206,
        "*3/*3": -2.3312,
        "Unknown": -0.2188,
    }.get(genotype, 0.0)


def vkorc1_coefficient(genotype):
    return {
        "A/A": -1.6974,
        "A/G": -0.8677,
        "Unknown": -0.4854,
    }.get(genotype, 0.0)


def bucket_weekly_dose(values):
    return pd.cut(
        values,
        bins=DOSE_BINS,
        labels=DOSE_LABELS,
        include_lowest=True,
    ).astype(int)


def iwpc_dose_buckets(frame, height_col, weight_col):
    vkorc1_col = "VKORC1 genotype: -1639 G>A (3673); chr16:31015190; rs9923231; C/T"
    race_clinical = frame["Race"].apply(race_coefficient_clinical)
    race_pharmacogenetic = frame["Race"].apply(race_coefficient_pharmacogenetic)
    cyp2c9 = frame["Cyp2C9 genotypes"].apply(cyp2c9_coefficient)
    vkorc1 = frame[vkorc1_col].apply(vkorc1_coefficient)

    clinical_sqrt_dose = (
        4.0376
        - 0.2546 * frame["Age__mode_decade"]
        + 0.0118 * frame[height_col]
        + 0.0134 * frame[weight_col]
        + race_clinical
        + 1.2799 * frame["Enzyme inducer status"]
        - 0.5695 * frame["Amiodarone (Cordarone)"]
    )
    pharmacogenetic_sqrt_dose = (
        5.6044
        - 0.2614 * frame["Age__mode_decade"]
        + 0.0087 * frame[height_col]
        + 0.0128 * frame[weight_col]
        + race_pharmacogenetic
        + 1.1816 * frame["Enzyme inducer status"]
        - 0.5503 * frame["Amiodarone (Cordarone)"]
        + cyp2c9
        + vkorc1
    )

    return {
        "Clinical": bucket_weekly_dose(clinical_sqrt_dose.pow(2)),
        "Pharmacogenetic": bucket_weekly_dose(pharmacogenetic_sqrt_dose.pow(2)),
    }


FEATURE_SET_HW_VARIANTS = {
    "v2_strict_knn_hw_full_dummies": {
        "height_weight_variant": "knn_hw",
        "height_col": "Height (cm)__knn_hw",
        "weight_col": "Weight (kg)__knn_hw",
    },
    "v2_strict_regression_hw_full_dummies": {
        "height_weight_variant": "regression_hw",
        "height_col": "Height (cm)__regression",
        "weight_col": "Weight (kg)__regression",
    },
}

baseline_accuracy_rows = []
for feature_set_name, variant in FEATURE_SET_HW_VARIANTS.items():
    predictions = iwpc_dose_buckets(
        df,
        height_col=variant["height_col"],
        weight_col=variant["weight_col"],
    )
    for baseline_name, predicted_bucket in predictions.items():
        correct = predicted_bucket.eq(df[TARGET_COL])
        baseline_accuracy_rows.append({
            "feature_set": feature_set_name,
            "height_weight_variant": variant["height_weight_variant"],
            "baseline": baseline_name,
            "correct_count": int(correct.sum()),
            "total_count": int(len(correct)),
            "accuracy": float(correct.mean()),
            "fraction_incorrect": float(1.0 - correct.mean()),
        })

baseline_accuracy = pd.DataFrame(baseline_accuracy_rows).sort_values(
    ["feature_set", "baseline"]
).reset_index(drop=True)
baseline_accuracy_path = OUTPUT_DIR / "warfarin_preprocess_v2_baseline_accuracy_by_hw_variant.csv"
baseline_accuracy.to_csv(baseline_accuracy_path, index=False)

print("Saved clinical/pharmacogenetic accuracy report:", baseline_accuracy_path)
baseline_accuracy

Saved clinical/pharmacogenetic accuracy report: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_baseline_accuracy_by_hw_variant.csv


,feature_set,height_weight_variant,baseline,correct_count,total_count,accuracy,fraction_incorrect
0,v2_strict_knn_hw_full_dummies,knn_hw,Clinical,3542,5528,0.640738,0.359262
1,v2_strict_knn_hw_full_dummies,knn_hw,Pharmacogenetic,3815,5528,0.690123,0.309877
2,v2_strict_regression_hw_full_dummies,regression_hw,Clinical,3533,5528,0.639110,0.360890
3,v2_strict_regression_hw_full_dummies,regression_hw,Pharmacogenetic,3821,5528,0.691208,0.308792


## Build V2 raw-clean and modeling tables

The raw-clean table preserves readable columns and engineered variants. The modeling table is numeric after one-hot encoding, with outcome/post-treatment leakage columns removed.

In [18]:
raw_clean_path = OUTPUT_DIR / "warfarin_preprocess_v2_raw_clean_table.csv"
df.to_csv(raw_clean_path, index=False)
print("Saved raw-clean V2 table:", raw_clean_path)
print("Raw-clean shape:", df.shape)

Saved raw-clean V2 table: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_raw_clean_table.csv
Raw-clean shape: (5528, 170)


In [19]:
# Columns that should not be one-hot encoded into the modeling table.
# We keep engineered replacements instead.
columns_to_drop_before_encoding = [
    *ID_COLUMNS,
    *POST_TREATMENT_OR_OUTCOME_COLUMNS,
    *FREE_TEXT_COLUMNS,
    "Age",
    "Age__mode_bucket",
    "Height (cm)",
    "Weight (kg)",
    "Indication for Warfarin Treatment",
    "Target INR",
    "Estimated Target INR Range Based on Indication",
]
columns_to_drop_before_encoding = [c for c in columns_to_drop_before_encoding if c in df.columns]

model_source = df.drop(columns=columns_to_drop_before_encoding, errors="ignore").copy()

# Track categorical dummy groups so feature sets can optionally drop one level per group.
categorical_cols = model_source.select_dtypes(include=["object", "category"]).columns.tolist()
category_levels = {
    col: sorted(model_source[col].dropna().astype(str).unique().tolist())
    for col in categorical_cols
}

model_encoded = pd.get_dummies(model_source, columns=categorical_cols, dummy_na=False, dtype=float)

# Ensure bool columns are numeric.
for col in model_encoded.columns:
    if model_encoded[col].dtype == bool:
        model_encoded[col] = model_encoded[col].astype(int)

# A useful intercept column for linear bandits.
model_encoded["Intercept"] = 1.0

modeling_table_path = OUTPUT_DIR / "warfarin_preprocess_v2_modeling_table.csv"
model_encoded.to_csv(modeling_table_path, index=False)

category_levels_path = OUTPUT_DIR / "warfarin_preprocess_v2_category_levels.json"
save_json(category_levels, category_levels_path)

print("Saved V2 modeling table:", modeling_table_path)
print("Saved category levels:", category_levels_path)
print("Modeling shape:", model_encoded.shape)
model_encoded.head()

Saved V2 modeling table: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_modeling_table.csv
Saved category levels: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_category_levels.json
Modeling shape: (5528, 307)


,Amiodarone (Cordarone),Carbamazepine (Tegretol),Phenytoin (Dilantin),Rifampin or Rifampicin,Therapeutic Dose of Warfarin,Therapeutic Dose of Warfarin__mg_week,Age__missing,Age__mode_decade,Amiodarone (Cordarone)__missing,Carbamazepine (Tegretol)__missing,Phenytoin (Dilantin)__missing,Rifampin or Rifampicin__missing,Enzyme inducer status,Indication__DVT,Indication__PE,Indication__Afib_or_flutter,Indication__Heart_Valve,Indication__Cardiomyopathy_or_LV_Dilation,Indication__Stroke,Indication__Post_Orthopedic,Indication__Other,Indication__Unknown,Indication__Multiple,Indication__Count,TargetINR__exact_missing,TargetINR__range_missing,TargetINR__any_available,TargetINR__value,TargetINR__range_lower,TargetINR__range_upper,TargetINR__range_midpoint,TargetINR__range_width,TargetINR__standard_2_to_3,TargetINR__high_intensity,TargetINR__low_intensity,ComorbidityText__missing,ComorbidityText__none_reported,Comorbidity__atrial_fibrillation_or_flutter,Comorbidity__heart_valve,Comorbidity__thrombosis_or_embolism,Comorbidity__cardiomyopathy_or_heart_failure,Comorbidity__diabetes,Comorbidity__hypertension,Comorbidity__hyperlipidemia,Comorbidity__cancer,Comorbidity__renal_or_kidney,Comorbidity__liver_or_hepatic,Comorbidity__stroke_or_cva,Comorbidity__obesity,Comorbidity__respiratory,Comorbidity__cardiovascular,MedicationText__missing,MedicationText__none_reported,MedicationText__aspirin,MedicationText__clopidogrel_or_antiplatelet,MedicationText__statin,MedicationText__amiodarone,MedicationText__acetaminophen,MedicationText__antibiotic,MedicationText__antifungal_azole,...,VKORC1 QC genotype: 2255C>T (7566); chr16:31011297; rs2359612; A/G_Unknown,VKORC1 genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C_A/A,VKORC1 genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C_A/C,VKORC1 genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C_C/C,VKORC1 genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C_Unknown,VKORC1 QC genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C_A/A,VKORC1 QC genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C_A/C,VKORC1 QC genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C_C/C,VKORC1 QC genotype: -4451 C>A (861); Chr16:31018002; rs17880887; A/C_Unknown,CYP2C9 consensus_*1/*1,CYP2C9 consensus_*1/*11,CYP2C9 consensus_*1/*13,CYP2C9 consensus_*1/*14,CYP2C9 consensus_*1/*2,CYP2C9 consensus_*1/*3,CYP2C9 consensus_*1/*5,CYP2C9 consensus_*1/*6,CYP2C9 consensus_*2/*2,CYP2C9 consensus_*2/*3,CYP2C9 consensus_*3/*3,CYP2C9 consensus_Unknown,VKORC1 -1639 consensus_A/A,VKORC1 -1639 consensus_A/G,VKORC1 -1639 consensus_G/G,VKORC1 -1639 consensus_Unknown,VKORC1 497 consensus_G/G,VKORC1 497 consensus_G/T,VKORC1 497 consensus_T/T,VKORC1 497 consensus_Unknown,VKORC1 1173 consensus_C/C,VKORC1 1173 consensus_C/T,VKORC1 1173 consensus_T/T,VKORC1 1173 consensus_Unknown,VKORC1 1542 consensus_C/C,VKORC1 1542 consensus_C/G,VKORC1 1542 consensus_G/G,VKORC1 1542 consensus_Unknown,VKORC1 3730 consensus_A/A,VKORC1 3730 consensus_A/G,VKORC1 3730 consensus_G/G,VKORC1 3730 consensus_Unknown,VKORC1 2255 consensus_C/C,VKORC1 2255 consensus_C/T,VKORC1 2255 consensus_T/T,VKORC1 2255 consensus_Unknown,VKORC1 -4451 consensus_A/A,VKORC1 -4451 consensus_A/C,VKORC1 -4451 consensus_C/C,VKORC1 -4451 consensus_Unknown,Age__bucket_unknown_10 - 19,Age__bucket_unknown_20 - 29,Age__bucket_unknown_30 - 39,Age__bucket_unknown_40 - 49,Age__bucket_unknown_50 - 59,Age__bucket_unknown_60 - 69,Age__bucket_unknown_70 - 79,Age__bucket_unknown_80 - 89,Age__bucket_unknown_90+,Age__bucket_unknown_Unknown,Intercept
0,0.0,0.0,0.0,0.0,1,49.0,0,6.0,0,1,1,1,0.0,0,0,0,0,0,0,1,0,0,0,1.0,0,1,1,2.5,2.5,2.5,2.5,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,0.0,1,42.0,0,5.0,0,1,1,1,0.0,

## Feature-set manifests

The modeling table intentionally contains more columns than any single experiment needs. The feature-set manifest lets later bandit notebooks select a named set of columns.

Every feature set excludes:

- true weekly dose;
- target label as a feature;
- stable-dose outcome column;
- INR on reported therapeutic dose;
- subject identifier and raw free-text fields.

In [20]:
TARGET_AND_NON_FEATURE_COLUMNS = {
    TARGET_COL,
    TRUE_DOSE_COL,
}

HEIGHT_WEIGHT_VARIANT_SUFFIXES = ["__knn_hw", "__regression_hw"]
REGRESSION_HW_BASE_COLUMNS = {"Height (cm)__regression", "Weight (kg)__regression"}


def is_height_weight_variant_column(col):
    col = str(col)
    return col in REGRESSION_HW_BASE_COLUMNS or any(col.endswith(suffix) for suffix in HEIGHT_WEIGHT_VARIANT_SUFFIXES)


def uses_selected_height_weight_variant(col, hw_variant):
    col = str(col)
    if hw_variant == "regression_hw":
        return col in REGRESSION_HW_BASE_COLUMNS or col.endswith("__regression_hw")
    if hw_variant == "knn_hw":
        return col.endswith("__knn_hw")
    raise ValueError(f"Unknown height/weight variant: {hw_variant}")


def base_feature_columns(hw_variant="regression_hw"):
    features = [col for col in model_encoded.columns if col not in TARGET_AND_NON_FEATURE_COLUMNS]
    features = [
        col for col in features
        if (not is_height_weight_variant_column(col)) or uses_selected_height_weight_variant(col, hw_variant)
    ]
    return features


feature_sets = {
    "v2_strict_regression_hw_full_dummies": base_feature_columns("regression_hw"),
    "v2_strict_knn_hw_full_dummies": base_feature_columns("knn_hw"),
}

feature_manifest = pd.DataFrame(
    {"feature_set": name, "feature_name": feature}
    for name, features in feature_sets.items()
    for feature in features
)
feature_summary = (
    feature_manifest.groupby("feature_set")
    .size()
    .reset_index(name="n_features")
    .sort_values("feature_set")
)

feature_manifest_path = OUTPUT_DIR / "warfarin_preprocess_v2_feature_sets.csv"
feature_summary_path = OUTPUT_DIR / "warfarin_preprocess_v2_feature_set_summary.csv"
feature_manifest.to_csv(feature_manifest_path, index=False)
feature_summary.to_csv(feature_summary_path, index=False)

print("Saved feature manifest:", feature_manifest_path)
print("Saved feature-set summary:", feature_summary_path)
feature_summary


Saved feature manifest: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_feature_sets.csv
Saved feature-set summary: /Users/dhillo/Garage/codeComplete/full measure/warfarin/output/preprocess_v2/warfarin_preprocess_v2_feature_set_summary.csv


,feature_set,n_features
0,v2_strict_knn_hw_full_dummies,295
1,v2_strict_regression_hw_full_dummies,295


## Sanity checks

In [21]:
# 1. No feature set should contain target/non-feature columns.
for feature_set_name, features in feature_sets.items():
    overlap = sorted(set(features) & TARGET_AND_NON_FEATURE_COLUMNS)
    assert not overlap, f"{feature_set_name} contains non-feature columns: {overlap}"

# 2. No feature set should contain post-treatment/outcome leakage columns or formula-derived predictions.
leakage_terms = [
    "Subject Reached Stable Dose of Warfarin",
    "INR on Reported Therapeutic Dose of Warfarin",
    TRUE_DOSE_COL,
    "ClinicalDose",
    "PharmacogeneticDose",
    "clinical_predicted",
    "pharmacogenetic_predicted",
]
for feature_set_name, features in feature_sets.items():
    leakage_hits = [feature for feature in features if any(term in feature for term in leakage_terms)]
    assert not leakage_hits, f"{feature_set_name} contains leakage/formula-derived columns: {leakage_hits[:5]}"

# 3. Each height/weight variant feature set should contain only its selected variant-specific columns.
for feature_set_name, features in feature_sets.items():
    wrong_variant_cols = [
        feature for feature in features
        if is_height_weight_variant_column(feature) and not uses_selected_height_weight_variant(feature, 'knn_hw' if 'knn' in feature_set_name else 'regression_hw')
    ]
    assert not wrong_variant_cols, f"{feature_set_name} contains wrong height/weight variant columns: {wrong_variant_cols[:5]}"

# 4. All feature-set columns should exist in the modeling table and be numeric.
for feature_set_name, features in feature_sets.items():
    missing = [feature for feature in features if feature not in model_encoded.columns]
    assert not missing, f"{feature_set_name} has missing columns: {missing[:5]}"
    non_numeric = model_encoded[features].select_dtypes(exclude=[np.number]).columns.tolist()
    assert not non_numeric, f"{feature_set_name} has non-numeric columns: {non_numeric[:5]}"

# 5. The fixed-formula report should cover both active feature sets and both requested baselines.
assert set(baseline_accuracy["feature_set"]) == set(feature_sets), "Baseline report feature sets do not match the manifest"
baseline_counts = baseline_accuracy.groupby("feature_set")["baseline"].nunique()
assert baseline_counts.eq(2).all(), "Each feature set must include Clinical and Pharmacogenetic results"

# 6. Modeling table should not contain NaNs.
remaining_nulls = int(model_encoded.isna().sum().sum())
assert remaining_nulls == 0, f"Modeling table still contains {remaining_nulls} missing values"

print("All sanity checks passed.")
print("Final modeling table shape:", model_encoded.shape)
print("Feature-set counts:")
print(feature_summary.to_string(index=False))


All sanity checks passed.
Final modeling table shape: (5528, 307)
Feature-set counts:
                         feature_set  n_features
       v2_strict_knn_hw_full_dummies         295
v2_strict_regression_hw_full_dummies         295


## How to load the cleaned V2 feature sets later

```python
modeling_df = pd.read_csv("output/preprocess_v2/warfarin_preprocess_v2_modeling_table.csv")
feature_sets = pd.read_csv("output/preprocess_v2/warfarin_preprocess_v2_feature_sets.csv")
feature_cols = feature_sets.loc[
    feature_sets["feature_set"] == "v2_strict_regression_hw_full_dummies",
    "feature_name",
].tolist()
X = modeling_df[feature_cols]
y = modeling_df["Therapeutic Dose of Warfarin"]
```

The active V2 feature manifest intentionally remains compact: regression height/weight and KNN height/weight variants. Both are raw-feature sets; neither includes clinical/pharmacogenetic formula-derived predictions.
